# Laboratorio 04.1 — Fundamentos de evaluación

Construiremos métricas deterministas para comparar respuestas con referencias y medir cumplimiento de un contrato estructurado.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Calcular exact match y accuracy sobre predicciones alineadas con referencias.
- Medir cumplimiento de esquema y conservar los fallos como resultados.
- Comparar dos sistemas usando el mismo conjunto de ejemplos.
- Guardar resultados reproducibles en DataFrames y CSV.

## 2. Conceptos

Evaluar significa definir una unidad de análisis, una referencia y una métrica que responda una pregunta. Exact match mide coincidencia estricta; accuracy mide proporción de etiquetas correctas. Cumplimiento de esquema mide forma, no verdad. Las métricas dependen de la calidad del conjunto de evaluación y no deben resumir dimensiones distintas en una sola cifra sin justificación.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Dataset etiquetado] --> B[Predicciones]
    B --> C[Exact match y accuracy]
    B --> D[Validacion Pydantic]
    C --> E[DataFrame de resultados]
    D --> E
    E --> F[CSV reproducible]
```

## 4. Preparación del entorno

Usamos pandas, Pydantic y las métricas locales compartidas por el curso. No se invoca ningún modelo.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [3]:
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from genai.environment import preparar_entorno
from genai.evaluation.metrics import (
    classification_accuracy, exact_match_rate, schema_compliance_rate,
)

raiz_curso = preparar_entorno()

## 5. Definición del problema

Comparamos etiquetas de intención para seis consultas breves de soporte. Dos sistemas simulados producen predicciones distintas sobre el mismo conjunto. Los datos son didácticos y la evaluación no hace afirmaciones sobre proveedores reales.

## 6. Línea base

Primero contamos coincidencias exactas para un único vector de predicciones. La línea base permite verificar a mano el numerador y el denominador antes de usar funciones reutilizables.

In [4]:
referencias = ['envios', 'devoluciones', 'pagos', 'cambios', 'envios', 'pagos']
predicciones_base = ['envios', 'pagos', 'pagos', 'cambios', 'envios', 'sin_clasificar']
coincidencias = [prediccion == referencia for prediccion, referencia in zip(predicciones_base, referencias)]
print(f'Exact match manual: {sum(coincidencias)}/{len(referencias)} = {sum(coincidencias) / len(referencias):.3f}')

Exact match manual: 4/6 = 0.667


## 7. Implementación

Usamos funciones comunes para evitar definiciones diferentes de la misma métrica. En producción conviene registrar también la versión del dataset, prompt, modelo y parámetros del experimento.

In [5]:
predicciones_candidato = ['envios', 'devoluciones', 'pagos', 'cambios', 'pagos', 'pagos']
resultados_sistemas = pd.DataFrame([
    {
        'sistema': nombre,
        'exact_match': exact_match_rate(predicciones, referencias),
        'accuracy': classification_accuracy(predicciones, referencias),
        'ejemplos': len(referencias),
    }
    for nombre, predicciones in (
        ('linea_base', predicciones_base),
        ('candidato', predicciones_candidato),
    )
])
display(resultados_sistemas)

,sistema,exact_match,accuracy,ejemplos
0,linea_base,0.666667,0.666667,6
1,candidato,0.833333,0.833333,6


Un resultado de texto también puede incumplir un contrato. Esta celda valida registros con un esquema mínimo; la aplicación debe registrar cada rechazo, no ocultarlo para subir artificialmente la tasa de éxito.

In [6]:
class RegistroRespuesta(BaseModel):
    model_config = ConfigDict(extra='forbid')

    categoria: str = Field(min_length=3)
    respuesta: str = Field(min_length=8)


def cumple_esquema(payload: dict[str, object]) -> bool:
    """Indica si un registro satisface el contrato Pydantic."""
    try:
        RegistroRespuesta.model_validate(payload)
        return True
    except ValidationError:
        return False

salidas = [
    {'categoria': 'envios', 'respuesta': 'El pedido sigue en tránsito.'},
    {'categoria': 'pagos', 'respuesta': 'OK'},
    {'categoria': 'cambios', 'respuesta': 'Puedes solicitar otra talla.'},
]
validez = [cumple_esquema(salida) for salida in salidas]
metricas_esquema = pd.DataFrame([{
    'schema_valid': schema_compliance_rate(validez),
    'registros_validos': sum(validez),
    'registros_totales': len(validez),
}])
display(metricas_esquema)

,schema_valid,registros_validos,registros_totales
0,0.666667,2,3


## 8. Experimentos

Mantenemos referencias fijas y cambiamos únicamente el sistema de predicción. El resumen de cada sistema incluye accuracy y cantidad de ejemplos; también guardamos resultados por ejemplo para localizar errores.

In [7]:
tabla_por_ejemplo = pd.DataFrame({
    'referencia': referencias,
    'linea_base': predicciones_base,
    'candidato': predicciones_candidato,
})
tabla_por_ejemplo['acierto_base'] = tabla_por_ejemplo['linea_base'] == tabla_por_ejemplo['referencia']
tabla_por_ejemplo['acierto_candidato'] = tabla_por_ejemplo['candidato'] == tabla_por_ejemplo['referencia']
display(tabla_por_ejemplo)
resultados_sistemas.to_csv(raiz_curso / 'outputs' / '04_01_system_metrics.csv', index=False)
tabla_por_ejemplo.to_csv(raiz_curso / 'outputs' / '04_01_example_results.csv', index=False)

,referencia,linea_base,candidato,acierto_base,acierto_candidato
0,envios,envios,envios,True,True
1,devoluciones,pagos,devoluciones,False,True
2,pagos,pagos,pagos,True,True
3,cambios,cambios,cambios,True,True
4,envios,envios,pagos,True,False
5,pagos,sin_clasificar,pagos,False,True


## 9. Evaluación

Las métricas responden preguntas distintas: accuracy resume etiquetas, exact match resume cadenas y cumplimiento de esquema verifica estructura. La similitud semántica compara embeddings de predicciones y referencias con cosine; ambos deben provenir del mismo encoder. El ejemplo siguiente usa vectores ilustrativos, no un modelo semántico preentrenado.

In [9]:
import numpy as np

from genai.evaluation.metrics import semantic_similarity_score

embeddings_prediccion = np.array([[0.98, 0.20], [0.10, 0.99]])
embeddings_referencia = np.array([[1.0, 0.0], [0.0, 1.0]])
similitud_media = semantic_similarity_score(
    embeddings_prediccion,
    embeddings_referencia,
)
tabla_similitud = pd.DataFrame([{
    'metodo': 'cosine_sobre_embeddings_ilustrativos',
    'similitud_semantica_media': similitud_media,
    'notas': 'Sustituye estos vectores por salidas del mismo encoder.',
}])
display(tabla_similitud)
tabla_similitud.to_csv(raiz_curso / 'outputs' / '04_01_semantic_similarity.csv', index=False)

,metodo,similitud_semantica_media,notas
0,cosine_sobre_embeddings_ilustrativos,0.987371,Sustituye estos vectores por salidas del mismo...


**Resumen de métricas.** Esta última celda consolida las métricas previamente calculadas para comparar sistemas; comprueba que la nueva medida semántica quede separada de exact match, accuracy y cumplimiento de esquema.

In [10]:
metricas_finales = resultados_sistemas.merge(
    pd.DataFrame([{'schema_valid': schema_compliance_rate(validez)}]),
    how='cross',
)
display(metricas_finales)
metricas_finales.to_csv(raiz_curso / 'outputs' / '04_01_evaluation_summary.csv', index=False)

,sistema,exact_match,accuracy,ejemplos,schema_valid
0,linea_base,0.666667,0.666667,6,0.666667
1,candidato,0.833333,0.833333,6,0.666667


## 10. Discusión

Un conjunto pequeño puede dar resultados inestables y sus etiquetas pueden ser ambiguas. La evaluación debe incluir ejemplos difíciles, segmentarse por categoría y conservar las salidas fallidas. El cumplimiento estructural no implica corrección semántica, seguridad ni seguimiento de instrucciones.

## 11. Ejercicios

1. Añade consultas ambiguas y define cómo evaluarlas.
2. Calcula accuracy por categoría y localiza la clase con más errores.
3. Introduce una salida con un campo adicional y compara modo permisivo y estricto.
4. Explica qué información se pierde al resumir todo en accuracy.

## 12. Desafío

Construye un conjunto de evaluación versionado con ejemplos de regresión. Añade una prueba automatizada que compare resultados contra una línea base y falle si una métrica crítica cae por debajo de un umbral acordado.

## 13. Ideas clave

- Cada métrica representa una pregunta distinta.
- Usa el mismo conjunto para comparar sistemas.
- Conserva resultados por ejemplo además de promedios.
- Forma válida y respuesta correcta son propiedades diferentes.